# Ruh Quran Morphology
Private, free-quota-only, bounded continuation of the same verified 46M V2 checkpoint and AdamW state. Original Quran, annotation and translation targets remain separately attributed. Candidate evaluation never automatically promotes production weights.


In [ ]:
import hashlib, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import torch
SESSION_STARTED = time.monotonic()
assert torch.cuda.is_available(), "Kaggle GPU is required"
REVISION = "37cdbd8452f252674d8a0e8a82a2d7fe46c96817"
OBJECTIVE = "morphology"
REPOSITORY = Path("/kaggle/working/mizan")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/CodeWithJuber/mizan.git", str(REPOSITORY)], check=True, timeout=90)
subprocess.run(["git", "fetch", "--depth=1", "origin", REVISION], cwd=REPOSITORY, check=True, timeout=90)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPOSITORY, check=True, timeout=30)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPOSITORY, text=True).strip() == REVISION
print({"source_revision": REVISION, "gpu": torch.cuda.get_device_name(0), "gpu_count": torch.cuda.device_count(), "torch": torch.__version__, "objective": OBJECTIVE})


In [ ]:
EXPECTED_FILES = {'config.json': '21cf2112caabe6225cbd85ee8927257819b81b046f92fecd94228dbde0920ebd', 'model.pt': 'fcbd2dac0a95513b5cfd7da665b178d2ec3bfd032d49cfcbf062061ed5712fac', 'tokenizer.json': '0899eaeee76dcfdd3c97f3afd61ce31158e7b795a26f1568d83727897e6cee85', 'vocab.json': 'b1f45d66b00e6e4e13f5799e10aeb1fef376fcde0fd108e23dcc10dbe0f92b4e'}
def hash_file(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
models = [p for p in Path("/kaggle/input").rglob("model.pt") if p.parent.name == "candidate" and p.parent.parent.name == "full-training-run"]
assert len(models) == 1, "Exactly one attached final full-training baseline is required"
BASELINE = models[0].parent
for filename, expected in EXPECTED_FILES.items():
    assert hash_file(BASELINE / filename) == expected, "Baseline hash mismatch: " + filename
assert json.loads((BASELINE / "config.json").read_text())["tokenizer_version"] == 2
assert (BASELINE.parent / "training_state.pt").is_file(), "Original AdamW state is required"
baseline_manifest = {"kernel_source": "zubairshaikh/ruh-full-training-repair", "known_kernel_version": 1, "files": EXPECTED_FILES, "optimizer_sha256": hash_file(BASELINE.parent / "training_state.pt")}
Path("/kaggle/working/baseline-manifest.json").write_text(json.dumps(baseline_manifest, indent=2))
DATA = Path("/kaggle/working/quranic-data")
subprocess.run([sys.executable, "-m", "scripts.prepare_quranic_data", "--cache", "/kaggle/working/quranic-cache", "--output", str(DATA)], cwd=REPOSITORY, check=True, timeout=240)
manifest = json.loads((DATA / "manifest.json").read_text())
assert manifest["coverage"]["verses"] == 6236 and manifest["coverage"]["surahs"] == 114


In [ ]:
remaining = 1720 - (time.monotonic() - SESSION_STARTED)
assert remaining > 600, "Insufficient session time; training not started"
training_budget = min(900, int(remaining - 540))
RUN = Path("/kaggle/working/quranic-run")
environment = dict(os.environ, OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
subprocess.run([sys.executable, "-m", "scripts.train_quranic_paths", "--data-dir", str(DATA), "--path", OBJECTIVE, "--resume-from", str(BASELINE), "--output", str(RUN), "--device", "cuda", "--steps", "20000", "--max-seconds", str(training_budget), "--batch-size", "2", "--precision", "float32"], cwd=REPOSITORY, env=environment, check=True, timeout=int(remaining - 240))
report = json.loads((RUN / "evaluation.json").read_text())
assert report["parameters"] == 45985563 and report["tokenizer_version"] == 2
assert report["promotion"]["approved"] is False
report.update(training_revision=REVISION, objective=OBJECTIVE, accelerator=torch.cuda.get_device_name(0), gpu_count=torch.cuda.device_count(), baseline_manifest=baseline_manifest, session_seconds=time.monotonic() - SESSION_STARTED)
(RUN / "evaluation.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
remaining = 1710 - (time.monotonic() - SESSION_STARTED)
assert remaining > 180, "Training saved; insufficient time for held-out generation"
subprocess.run([sys.executable, "-m", "scripts.generate_quranic_predictions", "--candidate", str(RUN / "candidate"), "--data-dir", str(DATA), "--path", OBJECTIVE, "--output-dir", str(RUN), "--device", "cuda", "--samples", "8"], cwd=REPOSITORY, env=environment, check=True, timeout=min(240, int(remaining - 120)))
print(json.dumps({"objective":OBJECTIVE,"steps":report["training"]["steps"],"held_out_before":report["held_out_before"],"held_out_after":report["held_out_after"],"promotion_approved":False}, indent=2))


In [ ]:
# Preserve the final model and optimizer, not redundant rolling snapshots.
files = [p for p in (RUN / "candidate").iterdir() if p.is_file()]
files += [p for p in RUN.iterdir() if p.is_file() and p.name != "latest-continuation.json"]
files += [p for p in DATA.rglob("*") if p.is_file()]
files += [Path("/kaggle/working/baseline-manifest.json")]
artifact_manifest = {str(p.relative_to("/kaggle/working")):{"bytes":p.stat().st_size,"sha256":hash_file(p)} for p in sorted(files)}
Path("/kaggle/working/quranic-artifact-manifest.json").write_text(json.dumps(artifact_manifest, indent=2))
archive_path = Path("/kaggle/working/quranic-candidate.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_STORED) as archive:
    for p in sorted(files):
        archive.write(p, arcname=str(p.relative_to("/kaggle/working")))
    archive.write("/kaggle/working/quranic-artifact-manifest.json", arcname="quranic-artifact-manifest.json")
# Read every archived file back; this also validates each member's CRC.
with zipfile.ZipFile(archive_path) as archive:
    for name, expected in artifact_manifest.items():
        h = hashlib.sha256()
        with archive.open(name) as handle:
            for block in iter(lambda:handle.read(1024 * 1024), b""):
                h.update(block)
        assert h.hexdigest() == expected["sha256"], "Archived artifact mismatch: " + name
backup = {"archive_sha256":hash_file(archive_path),"bytes":archive_path.stat().st_size,"files":len(artifact_manifest),"readback_verified":True,"source_revision":REVISION,"objective":OBJECTIVE,"session_seconds":time.monotonic()-SESSION_STARTED,"promotion_approved":False}
Path("/kaggle/working/backup-verification.json").write_text(json.dumps(backup, indent=2))
print(json.dumps(backup, indent=2))
